In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from PIL import Image
from predict_loop_functions import  predict_loop, random_images, noise_iterations
import kagglehub


In [2]:
import numpy as np
from PIL import Image
from pathlib import Path
from typing import List, Set, Optional

def get_random_images(directory: str, num_images: int, visited_folders: Optional[Set[str]] = None) -> List[str]:
    """
    Selects num_images unique images from a directory and its subfolders (infinite depth).
    
    Args:
        directory (str): Path to the root directory containing subfolders with images.
        num_images (int): Number of unique images to return.
        visited_folders (Optional[Set[str]]): Set to track folders containing selected images.
    
    Returns:
        List[str]: List of num_images unique image file paths.
        List[str]: List of folders visited previously (must be inputted by user in order to keep track of past usage).
    
    Raises:
        ValueError: If the directory is invalid, doesn't exist, or has fewer than num_images images.
    """
    # Convert directory to Path object and validate
    dir_path = Path(directory)
    if not dir_path.exists() or not dir_path.is_dir():
        raise ValueError(f"Directory '{directory}' does not exist or is not a directory")

    # Initialize visited_folders if None
    if visited_folders is None:
        visited_folders = set()

    # Define valid image extensions (case-insensitive)
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.gif'}

    # Recursively collect all image files from directory and all subdirectories
    image_paths = [
        str(p) for p in dir_path.rglob('*')
        if p.is_file() and p.suffix.lower() in image_extensions
    ]

    # Check if enough images are available
    if len(image_paths) < num_images:
        raise ValueError(
            f"Requested {num_images} images, but only {len(image_paths)} found in {directory}"
        )

    # Select num_images unique images randomly
    selected_indices = np.random.choice(
        len(image_paths), size=num_images, replace=False
    )
    selected_images = [image_paths[i] for i in selected_indices]

    return_images = np.empty((num_images, 144, 256))

    for index, selected_image in enumerate(selected_images):
        image = Image.open(selected_image)
        image = image.convert('L')
        if image.size != (256, 144): image = image.resize((256, 144))
        image = np.array(image)
        return_images[index] = image

    # Add parent folders of selected images to visited_folders
    for image_path in selected_images:
        parent_folder = str(Path(image_path).parent)
        visited_folders.add(parent_folder)

    return return_images, visited_folders

#### get random images and save

In [3]:
os.getcwd()

'c:\\Users\\joshf\\OneDrive\\GitHub\\fnn-haefner\\fnn\\input_noise'

In [4]:
imagenet_path = 'c:\\Users\\joshf\\Downloads\\imagenet-mini'
print("Path to dataset files:", imagenet_path)

Path to dataset files: c:\Users\joshf\Downloads\imagenet-mini


#### random images

In [5]:
import numpy as np
from PIL import Image
from pathlib import Path
from typing import List, Set, Optional

def get_random_images(directory: str, num_images: int, visited_folders: Optional[Set[str]] = None) -> List[str]:
    """
    Selects num_images unique images from a directory and its subfolders (infinite depth).
    
    Args:
        directory (str): Path to the root directory containing subfolders with images.
        num_images (int): Number of unique images to return.
        visited_folders (Optional[Set[str]]): Set to track folders containing selected images.
    
    Returns:
        List[str]: List of num_images unique image file paths.
        List[str]: List of folders visited previously (must be inputted by user in order to keep track of past usage).
    
    Raises:
        ValueError: If the directory is invalid, doesn't exist, or has fewer than num_images images.
    """
    # Convert directory to Path object and validate
    dir_path = Path(directory)
    if not dir_path.exists() or not dir_path.is_dir():
        raise ValueError(f"Directory '{directory}' does not exist or is not a directory")

    # Initialize visited_folders if None
    if visited_folders is None:
        visited_folders = set()

    # Define valid image extensions (case-insensitive)
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.gif'}

    # Recursively collect all image files from directory and all subdirectories
    image_paths = [
        str(p) for p in dir_path.rglob('*')
        if p.is_file() and p.suffix.lower() in image_extensions
    ]

    # Check if enough images are available
    if len(image_paths) < num_images:
        raise ValueError(
            f"Requested {num_images} images, but only {len(image_paths)} found in {directory}"
        )

    # Select num_images unique images randomly
    selected_indices = np.random.choice(
        len(image_paths), size=num_images, replace=False
    )
    selected_images = [image_paths[i] for i in selected_indices]

    return_images = np.empty((num_images, 144, 256))

    for index, selected_image in enumerate(selected_images):
        image = Image.open(selected_image)
        image = image.convert('L')
        if image.size != (256, 144): image = image.resize((256, 144))
        image = np.array(image)
        return_images[index] = image

    # Add parent folders of selected images to visited_folders
    for image_path in selected_images:
        parent_folder = str(Path(image_path).parent)
        visited_folders.add(parent_folder)

    return return_images, visited_folders

In [6]:
pics, folds = get_random_images(imagenet_path, 1000)

In [16]:
len(pics)

1000

In [8]:
type(pics[0])

numpy.ndarray

In [9]:
pics[0]

array([[228., 237., 238., ..., 134., 133., 130.],
       [215., 226., 237., ..., 134., 133., 132.],
       [213., 215., 225., ..., 136., 134., 134.],
       ...,
       [121., 121., 123., ...,  38.,  37.,  17.],
       [122., 121., 124., ...,  37.,  36.,  25.],
       [121., 119., 125., ...,  37.,  37.,  29.]])

In [37]:
np.min(pics[0])

np.float64(20.0)

In [38]:
np.max(pics[0])

np.float64(255.0)

#### predictions, august 8th, 2025

In [ ]:
# run using 1000 images from above
dbin, dbin_mean, dbin_var, dbin_rgns = predict_loop(noise_type = 'dynamic', images = pics, sigma = 0, scans = [[4,7]], 
                                   stochastic_bin_param = True)